## Import das bibliotecas

In [1]:
import gdown

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score
)

from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE

## Download do dados

In [ ]:


file_id = "12vm4oQEeH7ZqB6glXEPpkc5V91lQy0mk"
url = f"https://drive.google.com/uc?id={file_id}"
caminho_saida = "./data/credit_risk_dataset.csv"

# Baixa o arquivo ignorando os avisos de vírus do Drive
gdown.download(url, caminho_saida, quiet=False)

# Carrega no pandas
df = pd.read_csv(caminho_saida)

df.head()

Downloading...
From: https://drive.google.com/uc?id=12vm4oQEeH7ZqB6glXEPpkc5V91lQy0mk
To: /home/s050741159/Área de Trabalho/sctec_mlvc_c2_t3_m1_projeto_avaliativo/data/credit_risk_dataset.csv
100%|██████████| 1.80M/1.80M [00:03<00:00, 599kB/s]


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


## Análise Exploratória dos Dados

In [3]:
# Shape

df.shape

(32581, 12)

In [ ]:
# tipos de dados

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32581 non-null  int64  
 1   person_income               32581 non-null  int64  
 2   person_home_ownership       32581 non-null  str    
 3   person_emp_length           31686 non-null  float64
 4   loan_intent                 32581 non-null  str    
 5   loan_grade                  32581 non-null  str    
 6   loan_amnt                   32581 non-null  int64  
 7   loan_int_rate               29465 non-null  float64
 8   loan_status                 32581 non-null  int64  
 9   loan_percent_income         32581 non-null  float64
 10  cb_person_default_on_file   32581 non-null  str    
 11  cb_person_cred_hist_length  32581 non-null  int64  
dtypes: float64(3), int64(5), str(4)
memory usage: 3.0 MB


In [13]:
#valores unicos nas colunas categóricas

colunas_categoricas = ['person_home_ownership','loan_intent', 'loan_grade','cb_person_default_on_file']

print("Valores únicos das variáveis categóricas:\n")
for coluna in colunas_categoricas:
    print(f"{coluna}: {df[coluna].unique()}")

Valores únicos das variáveis categóricas:

person_home_ownership: <StringArray>
['RENT', 'OWN', 'MORTGAGE', 'OTHER']
Length: 4, dtype: str
loan_intent: <StringArray>
[         'PERSONAL',         'EDUCATION',           'MEDICAL',
           'VENTURE',   'HOMEIMPROVEMENT', 'DEBTCONSOLIDATION']
Length: 6, dtype: str
loan_grade: <StringArray>
['D', 'B', 'C', 'A', 'E', 'F', 'G']
Length: 7, dtype: str
cb_person_default_on_file: <StringArray>
['Y', 'N']
Length: 2, dtype: str


In [14]:
# visualização do balanceamento delas
for coluna in colunas_categoricas:
    print(f"\n🔹 {coluna.upper()} (Total únicos: {df[coluna].nunique()}):")
    freq = df[coluna].value_counts(dropna=False)
    perc = df[coluna].value_counts(dropna=False, normalize=True) * 100
    display(pd.DataFrame({'Contagem': freq, 'Porcentagem (%)': perc.round(2)}))



🔹 PERSON_HOME_OWNERSHIP (Total únicos: 4):


,Contagem,Porcentagem (%)
person_home_ownership,,
RENT,16446,50.48
MORTGAGE,13444,41.26
OWN,2584,7.93
OTHER,107,0.33



🔹 LOAN_INTENT (Total únicos: 6):


,Contagem,Porcentagem (%)
loan_intent,,
EDUCATION,6453,19.81
MEDICAL,6071,18.63
VENTURE,5719,17.55
PERSONAL,5521,16.95
DEBTCONSOLIDATION,5212,16.00
HOMEIMPROVEMENT,3605,11.06



🔹 LOAN_GRADE (Total únicos: 7):


,Contagem,Porcentagem (%)
loan_grade,,
A,10777,33.08
B,10451,32.08
C,6458,19.82
D,3626,11.13
E,964,2.96
F,241,0.74
G,64,0.20



🔹 CB_PERSON_DEFAULT_ON_FILE (Total únicos: 2):


,Contagem,Porcentagem (%)
cb_person_default_on_file,,
N,26836,82.37
Y,5745,17.63


In [15]:
#análise das variáveis numéricas

df.describe(include='number').round(2)

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_cred_hist_length
count,32581.00,32581.00,31686.00,32581.00,29465.00,32581.00,32581.00,32581.00
mean,27.73,66074.85,4.79,9589.37,11.01,0.22,0.17,5.80
std,6.35,61983.12,4.14,6322.09,3.24,0.41,0.11,4.06
min,20.00,4000.00,0.00,500.00,5.42,0.00,0.00,2.00
25%,23.00,38500.00,2.00,5000.00,7.90,0.00,0.09,3.00
50%,26.00,55000.00,4.00,8000.00,10.99,0.00,0.15,4.00
75%,30.00,79200.00,7.00,12200.00,13.47,0.00,0.23,8.00
max,144.00,6000000.00,123.00,35000.00,23.22,1.00,0.83,30.00


In [17]:
#balanceamento da variável alvo (loan_status)

df['loan_status'].value_counts(normalize=True)

loan_status
0    0.781836
1    0.218164
Name: proportion, dtype: float64